## Install


In [2]:
from google.colab import drive
drive.mount('/content/drive')
!pip install "pandas<2.0.0"
%pip install -r /content/drive/MyDrive/FPS/PEANUT/analysis/requirements.txt | grep -v 'already satisfied'

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 105.9 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 1.5.3 which is incompatible.
dask-cudf-cu12 25.2.2 requires pandas<2.2.4dev0,>=2.0, but you have pandas 1.5.3 which is incompatible.
cudf-cu12 25.2.1 requires pandas<2.2.4dev0,>=2.0, but you have pandas 1.5.3 which is incompatible.
dask-expr 1.1.21 requires pandas>=2, but you have pandas 1.5.3 which is incompatible.
xarray 2025.1.2 requires pandas>=2.1, but you have pandas 1.5.3 which is incompatible.
mizani 0.13.1 requires pandas>=2.2.0, but you have pandas 1.5.3 which is incompatible.
plotnine 0.1

## Imports


In [3]:
# pip install --upgrade numpy

In [1]:
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
from google.colab import files
from google.colab import auth


import numpy as np
# from ydata_profiling import ProfileReport
from typing import List, Dict
from summarytools import dfSummary
auth.authenticate_user()
print('Authenticated')

import joblib

%load_ext google.colab.data_table
from google.cloud import bigquery
import os
from google.auth import default
import gspread
import gspread_dataframe as gd
creds, _ = default()
# from pandas_profiling import ProfileReport

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

#testing bigframes
import bigframes.pandas as bpd

Authenticated


In [2]:
client = bigquery.Client(project="peanutproject-2024")

In [4]:
client

## Environment Data Loading

In [5]:
# microbiology_icu_charlson_lab_sofa_vitals_urinedf_not_imputed =joblib.load( '/content/drive/MyDrive/FPS/PEANUT/analysis/data_outputs/microbiology_icu_charlson_lab_sofa_vitals_urinedf_not_imputed.joblib')
# microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_not_imputed =joblib.load( '/content/drive/MyDrive/FPS/PEANUT/analysis/data_outputs/microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_not_imputed.joblib')

# Functions

In [6]:
def assign_block(df: pd.DataFrame) -> pd.Series:
    '''New column 'block' in the DataFrame df that contains the names of the blocks according to the conditions specified in the dictionary.
    Example: df['block'] = assign_block(df) '''
    blocks = []
    for index, row in df.iterrows():
        block_found= False
        for block, conditions in blocks_conditions.items():
            antibiotics, organisms = conditions
            if row['ab_name'].lower() in [ant.lower() for ant in antibiotics] and row['org_name'].lower() in [org.lower() for org in organisms]:
                blocks.append(block)
                block_found = True
                break
        if not block_found:
            blocks.append(np.nan)
    return blocks

In [7]:
#WIP
# def check_joblib_created (folder_id_name: str) -> list:
#         # Autenticación con Google Drive
#         auth.authenticate_user()
#         # Crear el servicio de Google Drive
#         drive_service = build('drive', 'v3')

#         # ID de la carpeta compartida en Google Drive
#         folder_id = folder_id_name #'1nT2-4yc4rduTte4RrSwYizwN_Ew5Eluy'

#         # Listar el contenido de la carpeta
#         list_of_dicts_files = drive_service.files().list(
#             q=f"'{folder_id}' in parents",
#             fields="files"
#         ).execute()
#         list_of_dicts_files_names = list_of_dicts_files.get('files', [])

#         return list_of_dicts_files_names

In [8]:
# check_joblib_created('14LIeFYOgJqk5Hvo-m-Lrps7z4T5AOQfD')

In [9]:
# Replace lists with [nan] by 'no_ab'
def replace_nan_list(value: List) -> List:
    if isinstance(value, list) and all(pd.isna(x) for x in value):
        return ['no_ab']
    return value

In [10]:
# Replace lists with nan by ['no_ab']
def replace_nan_list(value):
  if not isinstance(value, list) and pd.isna(value) :
        return ['no_ab']
  return value

In [11]:
# Function for translating drug names into active principles and ab_groups
def parse_active_pp_ab (drug_list: List):
    active_pp_list = []
    for drug in drug_list:
        if drug in ab_classification_ab_type_dict:
            active_pp_list.append(ab_classification_ab_type_dict[drug])
    return active_pp_list

In [12]:
def mice_imputation(data, max_iter=10, random_state=0):
    """
    Perform MICE imputation on the given DataFrame.

    Parameters:
    - data: pd.DataFrame, DataFrame containing missing values.
    - max_iter: int, Maximum number of imputation iterations.
    - random_state: int, Random seed for reproducibility.

    Returns:
    - pd.DataFrame, DataFrame with imputed values.
    """
    # Initialize the IterativeImputer
    imputer = IterativeImputer(max_iter=max_iter, random_state=random_state)

    # Fit and transform the data
    imputed_data = imputer.fit_transform(data)

    # Convert the result back to a DataFrame
    imputed_df = pd.DataFrame(imputed_data, columns=data.columns)

    return imputed_df

# Data loading

### SQL calls

In [13]:
# Ruta de la carpeta que contiene los archivos SQL
carpeta_sql = '/content/drive/MyDrive/FPS/PEANUT/analysis/sql_subqueries'

# Diccionario para almacenar los dataframes resultantes con nombres como claves
diccionario_dataframes_por_nombre = {}

# Iterar sobre los archivos SQL en la carpeta
for archivo_sql in os.listdir(carpeta_sql):
    if archivo_sql.endswith('.sql'):
        # Construir la ruta completa del archivo
        ruta_completa = os.path.join(carpeta_sql, archivo_sql)

        # Leer la consulta SQL desde el archivo
        with open(ruta_completa, 'r') as f:
            query_sql = f.read()
        client = bigquery.Client(project="peanutproject-2024")
        # Ejecutar la consulta y convertirla en un dataframe
        dataframe_resultante = client.query(query_sql).to_dataframe()

        # Utilizar el nombre del archivo como clave en el diccionario
        nombre_sin_extension = os.path.splitext(archivo_sql)[0]
        diccionario_dataframes_por_nombre[nombre_sin_extension] = dataframe_resultante


### Dataframes

Loading queries and extracting the data.

In [14]:
#main dataframe
icu_stays_with_microbiology_events = diccionario_dataframes_por_nombre['icu_stays_with_microbiology_events'] ## peanutproject-2024.stay_id_selection_peanut.stay_id_selection
#variables
vitals_sign = diccionario_dataframes_por_nombre['vitals_sign']
chemistry = diccionario_dataframes_por_nombre['chemistry']
coagulation = diccionario_dataframes_por_nombre['coagulation']
complete_blood_count = diccionario_dataframes_por_nombre['complete_blood_count']
diagnostics = diccionario_dataframes_por_nombre['diagnostics']
# age = diccionario_dataframes_por_nombre['age']
microbiology_events = diccionario_dataframes_por_nombre['microbiology_events']
icu = diccionario_dataframes_por_nombre['icu']
sofa = diccionario_dataframes_por_nombre['sofa']
charlson = diccionario_dataframes_por_nombre['charlson']
icu_complete = diccionario_dataframes_por_nombre['icu_complete']
microbiology_complete = diccionario_dataframes_por_nombre['microbiology_complete']
#septic shock
amines_inputevents = diccionario_dataframes_por_nombre['amines_inputevents']
amines_emar = diccionario_dataframes_por_nombre['amines_emar']
amines_pyxis = diccionario_dataframes_por_nombre['amines_pyxis']
angus_septic_shock = diccionario_dataframes_por_nombre['angus_septic_shock']
#previous stays, hospitalization
icu_stay_one_year_before = diccionario_dataframes_por_nombre['icu_stay_one_year_before']
hospitalization_90_days_before = diccionario_dataframes_por_nombre['hospitalization_90_days_before']
prev_icu_stay_same_admission = diccionario_dataframes_por_nombre['prev_icu_stay_same_admission']
#previous ab administration
previous_ab_inputevents = diccionario_dataframes_por_nombre['previous_ab_inputevents']
previous_ab_emar = diccionario_dataframes_por_nombre['previous_ab_emar']
previous_ab_pyxis = diccionario_dataframes_por_nombre['previous_ab_pyxis']
#admission location in hospital
adm_location_by_hadm_id = diccionario_dataframes_por_nombre['adm_location_by_hadm_id']
#oncologic patient
oncologic_patient = diccionario_dataframes_por_nombre['oncologic_patient']
#for counting previuos icus stays
number_prev_stays = diccionario_dataframes_por_nombre['number_prev_stays']
number_prev_stays_last_5_years = diccionario_dataframes_por_nombre['number_prev_stays_last_5_years']
#betalactamic_allergy
betalactamic_allergy = diccionario_dataframes_por_nombre['betalactamic_allergy']
#first weight measure on icu admission
first_weight_at_stay_admission = diccionario_dataframes_por_nombre[ 'first_weight_height_at_stay_admission']
#previous ab administration
previous_ab_inputevents_preemptive = diccionario_dataframes_por_nombre['previous_ab_inputevents_correct_treatment_hosp_before_result_6_blocks']
previous_ab_emar_preemptive = diccionario_dataframes_por_nombre['previous_ab_emar_correct_treatment_hosp_before_result_6_blocks']
previous_ab_pyxis_preemptive = diccionario_dataframes_por_nombre['previous_ab_pyxis_correct_treatment_hosp_before_result_6_blocks']

# Inclusion criteria

#### Microorganisms-antibiotics in blocks

In [15]:
condition1 = (
    (icu_stays_with_microbiology_events['org_name'].isin(['ESCHERICHIA COLI', 'PROTEUS MIRABILIS', 'ENTEROBACTER CLOACAE COMPLEX', 'KLEBSIELLA PNEUMONIAE', 'KLEBSIELLA OXYTOCA', 'ENTEROBACTER CLOACAE', 'ENTEROBACTER SPECIES'])) &
    (icu_stays_with_microbiology_events['ab_name'].isin(['CEFTRIAXONE', 'CEFEPIME', 'CEFTAZIDIME']))
)

condition2 = (
    (icu_stays_with_microbiology_events['org_name'].isin(['ESCHERICHIA COLI', 'PROTEUS MIRABILIS', 'ENTEROBACTER CLOACAE COMPLEX', 'KLEBSIELLA PNEUMONIAE', 'KLEBSIELLA OXYTOCA', 'ENTEROBACTER CLOACAE', 'ENTEROBACTER SPECIES'])) &
    (icu_stays_with_microbiology_events['ab_name'].isin(['MEROPENEM', 'IMIPENEM']))
)

condition3 = (
    (icu_stays_with_microbiology_events['org_name'].isin(['PSEUDOMONAS AERUGINOSA', 'ACINETOBACTER BAUMANNII COMPLEX', 'ACINETOBACTER BAUMANNII'])) &
    (icu_stays_with_microbiology_events['ab_name'].isin(['MEROPENEM', 'IMIPENEM']))
)

condition4 = (
    (icu_stays_with_microbiology_events['org_name'] == 'STENOTROPHOMONAS MALTOPHILIA') &
    (icu_stays_with_microbiology_events['ab_name'] == 'TRIMETHOPRIM/SULFA')
)

condition5 = (
    (icu_stays_with_microbiology_events['org_name'] == 'ENTEROCOCCUS FAECIUM') &
    (icu_stays_with_microbiology_events['ab_name'] == 'VANCOMYCIN')
)

condition6 = (
    (icu_stays_with_microbiology_events['org_name'].isin(['STAPH AUREUS COAG +', 'POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS', 'S. AUREUS POSITIVE; MRSA POSITIVE', 'S. AUREUS POSITIVE; MRSA NEGATIVE'])) &
    (icu_stays_with_microbiology_events['ab_name'] == 'OXACILLIN')
)

# Aplicar las condiciones y obtener el resultado filtrado
icu_stays_with_microbiology_events_filtered = icu_stays_with_microbiology_events[condition1 | condition2 | condition3 | condition4 | condition5 | condition6]

#### Greater than 16 years

In [16]:
microbiology_icu_filtered_df = icu_stays_with_microbiology_events_filtered[icu_stays_with_microbiology_events_filtered['admission_age']>16]

In [20]:
microbiology_icu_filtered_df['stay_id'].nunique()

6063

In [17]:
BB

NameError: name 'BB' is not defined

In [ ]:
microbiology_icu_filtered_df

# Data Wrangling

###  Merges by hadm_id

#### Charlson and diagnoses

In [ ]:
microbiology_icu_charlson_df = pd.merge(microbiology_icu_filtered_df,charlson, on='hadm_id')

#### Laboratory results

In [ ]:
##### Rename charttime
complete_blood_count = complete_blood_count.rename(columns={'charttime':'charttime_blood_count'})
coagulation = coagulation.rename(columns={'charttime':'charttime_coagulation'})
chemistry = chemistry.rename(columns={'charttime':'charttime_chemistry'})

In [ ]:
##### Variables selection
complete_blood_count_filtered_ordered = complete_blood_count[['hadm_id','charttime_blood_count','hematocrit', 'hemoglobin', 'mch', 'mchc', 'mcv', 'platelet', 'rbc','rdw', 'rdwsd', 'wbc']]
coagulation_filtered_ordered = coagulation[[ 'hadm_id','charttime_coagulation','d_dimer', 'fibrinogen', 'thrombin', 'inr', 'pt', 'ptt']]
chemistry_filtered_ordered = chemistry[['hadm_id','charttime_chemistry','albumin', 'globulin', 'total_protein', 'aniongap', 'bicarbonate', 'bun', 'calcium', 'chloride', 'creatinine', 'glucose', 'sodium', 'potassium']]
chemistry_filtered_ordered = chemistry.copy()

In [ ]:
#order by charttime and hadm_if for taking the nearest to microbiology tests
microbiology_icu_charlson_df_ordered = microbiology_icu_charlson_df.sort_values(by=['charttime', 'hadm_id'])
complete_blood_count_ordered = complete_blood_count_filtered_ordered.sort_values(by=['charttime_blood_count', 'hadm_id'])
coagulation_filtered_ordered = coagulation_filtered_ordered.sort_values(by=['charttime_coagulation', 'hadm_id'])
chemistry_filtered_ordered = chemistry_filtered_ordered.sort_values(by=['charttime_chemistry', 'hadm_id'])

In [ ]:
microbiology_icu_charlson_df_ordered =microbiology_icu_charlson_df_ordered.dropna(subset=['hadm_id'])
complete_blood_count_ordered = complete_blood_count_ordered.dropna(subset=['hadm_id'])
coagulation_filtered_ordered = coagulation_filtered_ordered.dropna(subset=['hadm_id'])
chemistry_filtered_ordered = chemistry_filtered_ordered.dropna(subset=['hadm_id'])

In [ ]:
microbiology_icu_charlson_lab_df = pd.merge_asof(microbiology_icu_charlson_df_ordered, complete_blood_count_ordered,
                           left_on='charttime', right_on='charttime_blood_count', by='hadm_id',
                           direction='backward')
microbiology_icu_charlson_lab_df = pd.merge_asof(microbiology_icu_charlson_lab_df, coagulation_filtered_ordered,
                           left_on='charttime', right_on='charttime_coagulation', by='hadm_id',
                           direction='backward')
microbiology_icu_charlson_lab_df = pd.merge_asof(microbiology_icu_charlson_lab_df, chemistry_filtered_ordered,
                           left_on='charttime', right_on='charttime_chemistry', by='hadm_id',
                           direction='backward')

In [ ]:
# check none negative differences, lab before microbiology test
# microbiology_icu_charlson_lab_df['time_difference_minutes'] = (microbiology_icu_charlson_lab_df['charttime'] - microbiology_icu_charlson_lab_df['charttime_chemistry']).dt.total_seconds() / 60
# microbiology_icu_charlson_lab_df[microbiology_icu_charlson_lab_df['time_difference_minutes'] < 0]

#### Septic shock - Amines- emar

In [ ]:
##### emar
microbiology_icu_charlson_lab_emar_df = pd.merge(microbiology_icu_charlson_lab_df, amines_emar, on=['hadm_id','cultive_charttime'], how='left', suffixes=('', '_remove'))

In [ ]:
microbiology_icu_charlson_lab_emar_df['amines_emar'] = [0 if pd.isna(i) else 1 for i in microbiology_icu_charlson_lab_emar_df['RowNum'] ]

In [ ]:
microbiology_icu_charlson_lab_emar_df['amines_emar_presc'] =  microbiology_icu_charlson_lab_emar_df['medication'].fillna('no_amines')
microbiology_icu_charlson_lab_emar_df = microbiology_icu_charlson_lab_emar_df.drop(columns=['medication'])
microbiology_icu_charlson_lab_emar_df['amines_emar_presc'] =  microbiology_icu_charlson_lab_emar_df['amines_emar_presc'].str.lower()

#### Septic shock -Angus

In [ ]:
microbiology_icu_charlson_lab_emar_df = pd.merge(microbiology_icu_charlson_lab_emar_df, angus_septic_shock, on=['hadm_id'], how='left', suffixes=('', '_remove'))

#### Ab - Emar - For ab before cultive and resulted R and previous treatments

In [ ]:
previous_ab_emar['hadm_id'] = previous_ab_emar['hadm_id'].astype(str)
microbiology_icu_charlson_lab_emar_df['hadm_id'] = microbiology_icu_charlson_lab_emar_df['hadm_id'].astype(str)
microbiology_icu_charlson_lab_emar_df_aux = pd.merge(microbiology_icu_charlson_lab_emar_df, previous_ab_emar, on=['hadm_id','cultive_charttime'], how='left', suffixes=('', '_remove'))


In [ ]:
# Group by stay_id to relate with the medication with the general_df
stayid_emaradmin_df = microbiology_icu_charlson_lab_emar_df_aux.groupby('stay_id').agg({
    'medication': lambda x: list(set(x))
}).reset_index()
##Add column with medication by stayid
microbiology_icu_charlson_lab_emar_df = pd.merge(microbiology_icu_charlson_lab_emar_df, stayid_emaradmin_df, on=['stay_id'], how='left', suffixes=('', '_remove'))
##Add ab_medication dicotonomus column
microbiology_icu_charlson_lab_emar_df['ab_emar'] = [0 if pd.isna(i[0]) else 1 for i in microbiology_icu_charlson_lab_emar_df['medication'] ]


In [ ]:
# Aplicar la función al DataFrame
microbiology_icu_charlson_lab_emar_df['medication'] = microbiology_icu_charlson_lab_emar_df['medication'].apply(replace_nan_list)
#rename medication to ab_emar_presc
microbiology_icu_charlson_lab_emar_df = microbiology_icu_charlson_lab_emar_df.rename(columns={'medication':'ab_emar_presc'})

In [ ]:
microbiology_icu_charlson_lab_emar_df['ab_emar_presc'] = microbiology_icu_charlson_lab_emar_df['ab_emar_presc'].apply(lambda x: [str(i).lower() if isinstance(i, str) else i for i in x] if isinstance(x, list) else x)

#### Ab - Emar - For  preemptive ab prescribed to the patient prior to the culture during the admission

In [ ]:
previous_ab_emar_preemptive['hadm_id'] = previous_ab_emar_preemptive['hadm_id'].astype(str)
microbiology_icu_charlson_lab_emar_df['hadm_id'] = microbiology_icu_charlson_lab_emar_df['hadm_id'].astype(str)
microbiology_icu_charlson_lab_emar_df_aux = pd.merge(microbiology_icu_charlson_lab_emar_df, previous_ab_emar_preemptive, on=['hadm_id','cultive_charttime'], how='left', suffixes=('', '_remove'))


In [ ]:
microbiology_icu_charlson_lab_emar_df_aux = microbiology_icu_charlson_lab_emar_df_aux.rename(columns={'medication':'emar_preemtive_medication'})

#### Ab - Inputevents -  For  preemptive ab prescribed to the patient prior to the culture during the admission

In [ ]:
previous_ab_inputevents_preemptive['hadm_id'] = previous_ab_inputevents_preemptive['hadm_id'].astype(str)
previous_ab_inputevents_preemptive = previous_ab_inputevents_preemptive.drop_duplicates()

In [ ]:
previous_ab_inputevents_preemptive = previous_ab_inputevents_preemptive.rename(columns={'itemid':'itemid_ab_inputevents_preemptive'})

In [ ]:
###aux merge to relate stay_id and itemid
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df_aux = pd.merge(microbiology_icu_charlson_lab_emar_df_aux, previous_ab_inputevents_preemptive, on=['hadm_id','cultive_charttime'], how='left', suffixes=('', '_remove'))
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df_aux['itemid_ab_inputevents_preemptive'] = np.where(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df_aux['itemid_ab_inputevents_preemptive'].isna(),None, microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df_aux['itemid_ab_inputevents_preemptive'])

In [ ]:
# Group by stay_id to relate with the medication with the general_df
hadm_ieadmin_df = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df_aux.groupby('hadm_id').agg({
    'itemid_ab_inputevents_preemptive': lambda x: list(set(x))
}).reset_index()
##delete stays without ie ab prescription
hadm_ieadmin_df = hadm_ieadmin_df[hadm_ieadmin_df['itemid_ab_inputevents_preemptive'].apply(lambda x: x != [None])]

In [ ]:
# Crear el diccionario de itemid a nombres de antibióticos que dan resultados de la consulta (en la consulta hay 13 pero solo dan 6 resultados)
itemid_to_ab = {
    225798: 'VANCOMYCIN',
    225851: 'CEFEPIME',
    225853: 'CEFTAZIDIME',
    225855: 'CEFTRIAXONE',
    225876: 'IMIPENEM/CILASTATIN',
    225883: 'MEROPENEM'
}
# Clean itemids and replace for the drug name and process to lower case
hadm_ieadmin_df['itemid_ab_inputevents_preemptive'] = hadm_ieadmin_df['itemid_ab_inputevents_preemptive'].apply(lambda x: [itemid_to_ab[item] if item in itemid_to_ab else item for item in x])
hadm_ieadmin_df['itemid_ab_inputevents_preemptive'] = hadm_ieadmin_df['itemid_ab_inputevents_preemptive'].apply(lambda x: [str(i).lower() if isinstance(i, str) else i for i in x] if isinstance(x, list) else x)

In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df = pd.merge(microbiology_icu_charlson_lab_emar_df_aux, hadm_ieadmin_df, on=['hadm_id'], how='left', suffixes=('', '_remove'))

In [ ]:
# Aplicar la función al DataFrame
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df['itemid_ab_inputevents_preemptive'] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df['itemid_ab_inputevents_preemptive'].apply(replace_nan_list)

In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df['ab_preemptive_ie'] = [1 if i[0] != 'no_ab' else 0 for i in microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df['itemid_ab_inputevents_preemptive'] ]

#### Ab - Pyxis -  For  preemptive ab prescribed to the patient prior to the culture during the admission

In [ ]:
##no coincidences with stays ids between tables and inclusion criterias
previous_ab_pyxis_preemptive['hadm_id'] = previous_ab_pyxis_preemptive['hadm_id'].astype(str)
previous_ab_pyxis_preemptive = previous_ab_pyxis_preemptive.rename(columns={'name':'name_ab_inputevents_pyxis'})
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df = pd.merge(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df, previous_ab_pyxis_preemptive, on=['hadm_id','cultive_charttime'], how='left', suffixes=('', '_remove'))
# microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['ab_preemptive_pyxis'] = [0 if pd.isna(i) else 1 for i in microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['name_ab_inputevents_pyxis'] ]

#### Hospitalization unit

In [ ]:
#
adm_location_by_hadm_id['hadm_id'] = adm_location_by_hadm_id['hadm_id'].astype(str)
microbiology_icu_charlson_lab_emar_df = pd.merge(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df, adm_location_by_hadm_id, on=['hadm_id'], how='left')

#### Oncologic patients

In [ ]:
oncologic_patient['hadm_id'] = oncologic_patient['hadm_id'].astype(str)
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['hamd_id'] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['hadm_id'].astype(str)
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df = pd.merge(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df, oncologic_patient, on='hadm_id', how='left', suffixes=('', '_remove'))

###  Merges by stay_id

#### SOFA

In [ ]:
sofa['stay_id'] = sofa['stay_id'].astype(int)
sofa = sofa.rename(columns={'starttime':'starttime_sofa'})
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df = pd.merge(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df, sofa, on=['stay_id','cultive_charttime'], how='left', suffixes=('', '_remove'))

#### Vitals table

In [ ]:
vitals_sign['stay_id'] = vitals_sign['stay_id'].astype(str)
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['stay_id'] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['stay_id'].astype(str)

In [ ]:
##### Rename charttime
vitals_sign = vitals_sign.rename(columns={'charttime':'charttime_vitalsign'})

In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df = pd.merge(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df, vitals_sign, on=['stay_id','cultive_charttime'], how='left', suffixes=('', '_remove'))

#### Amines - Inputevents

In [ ]:
amines_inputevents['stay_id'] = amines_inputevents['stay_id'].astype(str)
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df = pd.merge(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df, amines_inputevents, on=['stay_id','cultive_charttime'], how='left', suffixes=('', '_remove'))
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df['amines_inputevents'] = [0 if pd.isna(i) else 1 for i in microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df['itemid'] ]

In [ ]:
itemid_to_amine = {
    221662: 'DOPAMINE',
    221289: 'EPINEPHRINE',
    221653: 'DOBUTAMINE',
    221906: 'NOREPINEPHRINE',
    221749: 'PHENYLEPHRINE',
    222315: 'VASOPRESSIN'
}

# Crear la nueva columna amines_ie_presc
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df['amines_ie_presc'] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df['itemid'].map(itemid_to_amine).fillna('no_amines')
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df['amines_ie_presc'] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df['amines_ie_presc'].str.lower()

In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df.drop(columns=['itemid'])

#### Amines - Pyxis

In [ ]:
##no coincidences with stays ids between tables and inclusion criterias
amines_pyxis['stay_id'] = amines_pyxis['stay_id'].astype(str)
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df = pd.merge(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df, amines_pyxis, on=['stay_id','cultive_charttime'], how='left', suffixes=('', '_remove'))
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['amines_pyxis'] = [0 if pd.isna(i) else 1 for i in microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['name'] ]

#### Ab - Pyxis -  For ab before cultive and resulted R and previous treatments

In [ ]:
##no coincidences with stays ids between tables and inclusion criterias
previous_ab_pyxis['stay_id'] = previous_ab_pyxis['stay_id'].astype(str)
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df = pd.merge(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df, previous_ab_pyxis, on=['stay_id','cultive_charttime'], how='left', suffixes=('', '_remove'))
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['ab_pyxis'] = [0 if pd.isna(i) else 1 for i in microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['name'] ]

#### Ab - Inputevents -  For ab before cultive and resulted R and previous treatments

In [ ]:
previous_ab_inputevents['stay_id'] = previous_ab_inputevents['stay_id'].astype(str)
previous_ab_inputevents = previous_ab_inputevents.drop_duplicates()

In [ ]:
###aux merge to relate stay_id and itemid
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df_aux = pd.merge(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df, previous_ab_inputevents, on=['stay_id','cultive_charttime'], how='left', suffixes=('', '_remove'))
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df_aux['itemid'] = np.where(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df_aux['itemid'].isna(),None, microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df_aux['itemid'])

In [ ]:
# Group by stay_id to relate with the medication with the general_df
stayid_ieadmin_df = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_df_aux.groupby('stay_id').agg({
    'itemid': lambda x: list(set(x))
}).reset_index()
##delete stays without ie ab prescription
stayid_ieadmin_df = stayid_ieadmin_df[stayid_ieadmin_df['itemid'].apply(lambda x: x != [None])]

In [ ]:
# Crear el diccionario de itemid a nombres de antibióticos
itemid_to_ab = {
    225662: 'ZGENTAMICIN (PEAK)',
    226068: 'ZGENTAMICIN (TROUGH)',
    226069: 'ZGENTAMICIN (RANDOM)',
    227447: 'GENTAMICIN (RANDOM)',
    227448: 'GENTAMICIN (PEAK)',
    227449: 'GENTAMICIN (TROUGH)',
    225840: 'AMIKACIN',
    225842: 'AMPICILLIN',
    225843: 'AMPICILLIN/SULBACTAM (UNASYN)',
    225845: 'AZITHROMYCIN',
    225847: 'AZTREONAM',
    225850: 'CEFAZOLIN',
    225851: 'CEFEPIME',
    225853: 'CEFTAZIDIME',
    225855: 'CEFTRIAXONE',
    225859: 'CIPROFLOXACIN',
    225860: 'CLINDAMYCIN',
    225875: 'GENTAMICIN',
    225879: 'LEVOFLOXACIN',
    225881: 'LINEZOLID',
    225883: 'MEROPENEM',
    225884: 'METRONIDAZOLE',
    225886: 'MOXIFLOXACIN',
    225888: 'NAFCILLIN',
    225899: 'BACTRIM (SMX/TMP)',
    227691: 'KEFLEX',
    229059: 'CHLORAMPHENICOL',
    229587: 'CEFTAROLINE',
    226403: 'GU IRRIGANT - AMPHOTERICIN B'
}

# Clean itemids and replace for the drug name and process to lower case
stayid_ieadmin_df['itemid'] = stayid_ieadmin_df['itemid'].apply(lambda x: [itemid_to_ab[item] if item in itemid_to_ab else item for item in x])
stayid_ieadmin_df['itemid'] = stayid_ieadmin_df['itemid'].apply(lambda x: [str(i).lower() if isinstance(i, str) else i for i in x] if isinstance(x, list) else x)

In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df = pd.merge(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df, stayid_ieadmin_df, on=['stay_id'], how='left', suffixes=('', '_remove'))
#rename medication to ab_ie_presc
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.rename(columns={'itemid':'ab_ie_presc'})

In [ ]:
# Aplicar la función al DataFrame
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['ab_ie_presc'] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['ab_ie_presc'].apply(replace_nan_list)

In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['ab_inputevents'] = [1 if i[0] != 'no_ab' else 0 for i in microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['ab_ie_presc'] ]

#### Antonio: Number of previous icu stays las 5 years
*   Dec 2024: Changed to 5 years before
*   SQL query is by subject_id



In [ ]:
number_prev_stays_last_5_years['stay_id'] = number_prev_stays_last_5_years['stay_id'].astype(str)

In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df = pd.merge(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df, number_prev_stays_last_5_years, on=['stay_id'], how='left', suffixes=('', '_remove'))

In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['previous_icu_stays_5_years'].value_counts()

#### Weight and height

In [ ]:
first_weight_at_stay_admission['stay_id'] = first_weight_at_stay_admission['stay_id'].astype(str)

In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df = pd.merge(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df, first_weight_at_stay_admission, on=['stay_id'], how='left', suffixes=('', '_remove'))

### Merges by subject_id

#### Betalactamic allergy

In [ ]:
betalactamic_allergy['hadm_id'] = betalactamic_allergy['hadm_id'].astype('str')

In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df = pd.merge(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df, betalactamic_allergy, on=['hadm_id'], how='left', suffixes=('', '_remove'))

## New variables

### BMI

In [ ]:
# Assuming your DataFrame is named microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df
# and it contains columns 'weight' and 'height'

# Convert weight and height to numeric, handling potential errors
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['weight_admit'] = pd.to_numeric(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['weight_admit'], errors='coerce')
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['height'] = pd.to_numeric(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['height'], errors='coerce')

# Calculate BMI (weight (kg) / (height (m))^2)
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['BMI'] = (
    microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['weight_admit'] /
    (microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['height'] / 100) ** 2
)

### Sample  origin

In [ ]:
### origin column
origin_conditions = {
    'Respiratory sample': (['Mini-BAL', 'SPUTUM','BRONCHOALVEOLAR LAVAGE','BRONCHIAL WASHINGS','ASPIRATE','TRACHEAL ASPIRATE']),
    'Blood sample': (['BLOOD CULTURE', 'FLUID RECEIVED IN BLOOD CULTURE BOTTLES','BLOOD CULTURE ( MYCO/F LYTIC BOTTLE)']),
    'Urine sample': (['URINE' ,'URINE,KIDNEY']),
    'Abdominal liquid sample': (['BILE' ,'PERITONEAL FLUID']),
    'Venous catheter': (['CATHETER TIP-IV']),
    'Other devices': (['FOREIGN BODY','Foreign Body - Sonication Culture']),
    'Pleural fluid sample': (['PLEURAL FLUID']),
    'Synovial fluid sample': (['JOINT FLUID']),
    'Spinal fluid sample': (['CSF;SPINAL FLUID']),
    'Faeces sample': (['STOOL'])
}

microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['sample_origin'] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.apply(lambda row: next((key for key, values in origin_conditions.items() if row['spec_type_desc'] in values[0]), row['spec_type_desc']), axis=1)


### Dates of admission

In [ ]:
### Variables found in article
### https://repositoriosaludmadrid.es/bitstream/20.500.12530/54340/1/IJIMAI.pdf
# microbiology_icu_charlson_lab_sofa_vitals_urinedf['year_icu_admission'] = microbiology_icu_charlson_lab_sofa_vitals_urinedf['icu_intime'].dt.year
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['day_of_week_icu_admission'] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['icu_intime'].dt.day_name()
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['month_icu_admission'] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['icu_intime'].dt.month_name()
# microbiology_icu_charlson_lab_sofa_vitals_urinedf['year_culture_request']  = microbiology_icu_charlson_lab_sofa_vitals_urinedf['charttime'].dt.year
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['day_of_week_culture_request'] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['cultive_charttime'].dt.day_name()
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['month_culture_request']  = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['cultive_charttime'].dt.month_name()

### Blocks

In [ ]:
### Block column
blocks_conditions = {
    'Bloque 1': (['CEFTRIAXONE', 'CEFEPIME', 'CEFTAZIDIME'], ['ESCHERICHIA COLI', 'PROTEUS MIRABILIS', 'ENTEROBACTER CLOACAE COMPLEX', 'KLEBSIELLA PNEUMONIAE', 'KLEBSIELLA OXYTOCA',  'ENTEROBACTER CLOACAE', 'ENTEROBACTER SPECIES']),
    'Bloque 2': (['MEROPENEM', 'IMIPENEM'], ['ESCHERICHIA COLI' , 'PROTEUS MIRABILIS', 'ENTEROBACTER CLOACAE COMPLEX' , 'KLEBSIELLA PNEUMONIAE', 'KLEBSIELLA OXYTOCA', 'ENTEROBACTER CLOACAE' , 'ENTEROBACTER SPECIES']),
    'Bloque 3': ( ['MEROPENEM', 'IMIPENEM'], ['PSEUDOMONAS AERUGINOSA', 'ACINETOBACTER BAUMANNII COMPLEX', 'ACINETOBACTER BAUMANNII']),
    'Bloque 4': (['TRIMETHOPRIM/SULFA'], ['STENOTROPHOMONAS MALTOPHILIA']),
    'Bloque 5': (['VANCOMYCIN'], ['ENTEROCOCCUS FAECIUM']),
    'Bloque 6': (['OXACILLIN'], ['STAPH AUREUS COAG +', 'POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS', 'S. AUREUS POSITIVE; MRSA POSITIVE','S. AUREUS POSITIVE; MRSA NEGATIVE'])
}
# Aplicar la función para asignar bloques al DataFrame completo
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['block'] = assign_block(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df)


### Previous ab prescription

In [ ]:
### AB administered previous resistance
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['has_previous_ab'] = (
    microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['ab_inputevents'] +
    microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['ab_emar'] +
    microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['ab_pyxis']
).apply(lambda x: 1 if x >= 1 else 0)

In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.drop(columns=['ab_inputevents','ab_emar','ab_pyxis'])

In [ ]:
### Antibiotic group and Active PP labelling
#### We will join the df of each ab_emar_presc, ab_ie_presc, ab_pyxis_presc we will remove the repeated ones after that we will find their groups and their PA in the antibiotic_in_dataset.joblib and we will make dummies

In [ ]:
ab_classification =joblib.load( '/content/drive/MyDrive/FPS/PEANUT/analysis/excels_classifications/antibiotics_in_dataset.joblib')

In [ ]:
# Join columns with prvious ab treatments deleting repeated values
def union_unique_lists(row):
    set_union = set(row['ab_emar_presc']) | set(row['ab_ie_presc'])
    if len(set_union) > 1 and 'no_ab' in set_union:
        set_union.remove('no_ab')
    return list(set_union)

# Apply function
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['ab_presc_total'] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.apply(union_unique_lists, axis=1)

In [ ]:
#dict creation linking ab_name with active_principle and type for merging with the main table
ab_classification_ab_type_dict = dict(zip(ab_classification['ab_name'],ab_classification['ab_type']))
ab_classification_ab_pa_dict = dict(zip(ab_classification['ab_name'],ab_classification['active_principle']))

In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['group_ab'] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['ab_presc_total'].apply(parse_active_pp_ab)

In [ ]:
# Get all unique values present in the lists
unique_values = set()
for lista in microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['group_ab']:
    unique_values.update(lista)

# Create dummy columns for each unique value
for value in unique_values:
    microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['ab_group_'+value] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['group_ab'].apply(lambda lista: 1 if value in lista else 0)

In [ ]:
##### by active principle
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['active_principle'] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['ab_presc_total'].apply(parse_active_pp_ab)

In [ ]:
# Get all unique values present in the lists
unique_values = set()
for lista in microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['active_principle']:
    unique_values.update(lista)

# Create dummy columns for each unique value
for value in unique_values:
    microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['active_principle'+value] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['active_principle'].apply(lambda lista: 1 if value in lista else 0)

### Time in hospital before suspicion

In [ ]:
# Time from admission to suspicion
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['time_in_hospital'] = (pd.to_datetime(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['charttime']) - pd.to_datetime(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['admittime'])).dt.total_seconds()/60

In [ ]:
### less than 24hours in hospital - 1440 minutes
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['time_in_hospital_less_24h'] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['time_in_hospital'].apply(lambda x: 1 if x < 1440 else 0)

In [ ]:
# # check
# microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df[microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['subject_id']==13645451]
# df_min_stay_id[df_min_stay_id['subject_id']==13645451]

### Oncologic patient

In [ ]:
#oncologic patient
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['oncologic_patient'] = 0
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['oncologic_patient'] = np.where(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['icd_code'].isna(),0,1)

In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['oncologic_patient'].value_counts()

### Hosp 90 days before

In [ ]:
#### Previous hospital admission - window: 90 days
list_of_hosp_90_days_before = hospitalization_90_days_before['hadm_id'].astype(str)
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['hadm_id'] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['hadm_id'].astype(str)
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['hosp_before_90_days'] = 0
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.loc[microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['hadm_id'].isin(list_of_hosp_90_days_before), 'hosp_before_90_days'] = 1

### ICUs stays same admission

In [ ]:
### Previous icu stay in the same admission
list_of_stays_with_before_same_admission = prev_icu_stay_same_admission['stay_id'].astype(str)
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['stay_before_same_admission'] = 0
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.loc[microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['stay_id'].isin(prev_icu_stay_same_admission), 'stay_before_same_admission'] = 1

### ICU stay year before

In [ ]:
### Previous icu stay - window: until 1 year before to this admission
list_of_stays_one_year_before = icu_stay_one_year_before['stay_id'].astype(str)
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['stay_before'] = 0
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.loc[microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['stay_id'].isin(list_of_stays_one_year_before), 'stay_before'] = 1

### Number of ICU stays 5 years before

Resulted in the join

### Resistance 1 year before

In [ ]:
### Previous resistance in general: window 1 year -
# Convert the 'charttime' column to datetime type
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['cultive_charttime'] = pd.to_datetime(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['cultive_charttime'])

# Sort the DataFrame by 'subject_id' and 'charttime'
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.sort_values(by=['subject_id', 'cultive_charttime'])

# Create a new column to identify if there has been a culture with result 'R' at least one year before
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['had_R_before_1_year'] = False
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['had_R_before_block'] = 'no_prev_R'

# Iterate over each subject_id
for subject in microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['subject_id'].unique():
    # Filter the DataFrame by the current subject_id
    subject_microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df[microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['subject_id'] == subject]
  # Obtener las fechas de cultivos con resultado 'R'
    r_rows = subject_microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df[subject_microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['interpretation'] == 'R'][['cultive_charttime', 'block']]

    # Iterar sobre las filas de cada sujeto
    for index, row in subject_microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.iterrows():
        # Filtrar las filas con resultado 'R' dentro del período de un año antes de la fila actual
        relevant_r_rows = r_rows[(0 < (row['cultive_charttime'] - r_rows['cultive_charttime']).dt.days) &
                                 ((row['cultive_charttime'] - r_rows['cultive_charttime']).dt.days <= 365)]
        if not relevant_r_rows.empty:
            microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.loc[index, 'had_R_before_1_year'] = True
            microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.loc[index, 'had_R_before_block'] = ', '.join(relevant_r_rows['block'].unique())

#EXAMPLE_TO_CHECK
# microbiology_icu_charlson_lab_sofa_vitals_df[( microbiology_icu_charlson_lab_sofa_vitals_df['subject_id']==10004401) & ( microbiology_icu_charlson_lab_sofa_vitals_df['interpretation']=='R')][['interpretation','charttime']]

### Resistance same block before (1 year)

In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['had_R_before_sameblock'] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.apply(
    lambda row: row['block'] in row['had_R_before_block'],
    axis=1
)

In [ ]:
# check
# microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df[microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['had_R_before_sameblock']==True][['block','had_R_before_block']]

In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['had_R_before_block'] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['had_R_before_block'].apply(lambda x: x.split(', ') if x else [])

In [ ]:
# Check the number of unique values in 'had_R_before_block' and create the new column accordingly
def check_length(value):
    return 'MoreThanOne' if len(value) > 1 else value[0]

# Apply the function to create the new column
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['had_R_before_block_aggregated'] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['had_R_before_block'].apply(check_length)


In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['had_R_before_block_aggregated']

In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['had_R_before_block'].value_counts()

### Antonio: Resistance 90 days before

In [ ]:
### Previous resistance in general: window 90 days -
# Convert the 'charttime' column to datetime type
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['cultive_charttime'] = pd.to_datetime(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['cultive_charttime'])

# Sort the DataFrame by 'subject_id' and 'charttime'
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.sort_values(by=['subject_id', 'cultive_charttime'])

# Create a new column to identify if there has been a culture with result 'R' at least 90 days before
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['had_R_before_90_days'] = False

# Iterate over each subject_id
for subject in microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['subject_id'].unique():
    # Filter the DataFrame by the current subject_id
    subject_microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df[microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['subject_id'] == subject]
  # Obtener las fechas de cultivos con resultado 'R'
    r_rows = subject_microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df[subject_microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['interpretation'] == 'R'][['cultive_charttime', 'block']]

    # Iterar sobre las filas de cada sujeto
    for index, row in subject_microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.iterrows():
        # Filtrar las filas con resultado 'R' dentro del período de un año antes de la fila actual
        relevant_r_rows = r_rows[(0 < (row['cultive_charttime'] - r_rows['cultive_charttime']).dt.days) &
                                 ((row['cultive_charttime'] - r_rows['cultive_charttime']).dt.days <= 90)]
        if not relevant_r_rows.empty:
            microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.loc[index, 'had_R_before_90_days'] = True

In [ ]:
##TO_CHECK AND WITH MORE THAN ONE BLOCK RESISTANCE
# microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df[microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df ['subject_id']==18965721]
# microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df[microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df ['subject_id']==19585869]

In [ ]:
# ##check stays with more than one previous resistance with different blocks
# filtered_df = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df[microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['interpretation'] == 'R']

# # Agrupa por 'stays_id' y filtra los grupos que tienen más de un valor único en la columna 'block'
# multiple_blocks_df = filtered_df.groupby('stay_id').filter(lambda x: x['block'].nunique() > 1)

# # Obtén los stays_id únicos que cumplen con la condición
# stays_id_with_multiple_blocks = multiple_blocks_df['stay_id'].unique()

### Septic shock

In [ ]:
# Septic Shock
## Amines 6 hours before or after the suspicious
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['has_septic_shock'] = (
    microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['amines_emar'] +
    microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['amines_inputevents'] +
    microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['angus']
    # +
    # microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['amines_pyxis'] #without data
).apply(lambda x: 1 if x >= 1 else 0)

In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.drop(columns=['amines_emar','amines_inputevents','angus'])

In [ ]:
septic_shock_counts = (
    microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df
    .groupby('stay_id')['has_septic_shock']
    .sum()
    .reset_index()
)
septic_shock_counts[septic_shock_counts['has_septic_shock'] > 0]

### Betalactamic allergy

In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['betalactamic_allergy'].value_counts()

### Antonio: Same antibiotic prescribed to the patient prior to the culture result Resistant
Indicates whether the same antibiotic was prescribed to the patient prior to the culture and resulted resistant


In [ ]:
# Función para verificar si el nombre de ab_name (en mayúsculas) está en la lista de ab_ie_presc (en minúsculas)
def is_in_list(row):
    # Verificamos si ab_name y ab_ie_presc no son None
    if row['ab_name'] is not None and row['ab_ie_presc'] is not None:
        # Convertimos a minúsculas solo si ab_name no es None
        name = row['ab_name'].lower() if row['ab_name'] else None
        # Revisamos si name está en la lista de ab_ie_presc, asegurándonos de que todos los elementos también estén en minúsculas
        if name:
            return name in [x.lower() for x in row['ab_ie_presc'] if x is not None]
    return False  # Si alguno es None o no cumple las condiciones, devolvemos False

# Aplicamos la función al DataFrame
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['received_same_ab_before_test_str'] = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.apply(is_in_list, axis=1)

In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df[(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['received_same_ab_before_test_str']==True) & (microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['interpretation']=='R')][['ab_name']].value_counts()

In [ ]:
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df[(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['received_same_ab_before_test_str']==True) & (microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['interpretation']=='S')][['ab_name']].value_counts()

In [ ]:
###Si alguna de received_same_ab_before_test_str en esa estancia es True, entonces

# microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols['received_same_ab_before_test_str'] = (
#     microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols
#     .groupby('stay_id')['received_same_ab_before_test_str']
#     .transform('max')
# )

In [ ]:
pd.set_option('display.max_columns', None)

### DROPPED Antonio: Correct preemptive antibiotic prescribed to the patient prior to the culture
Indicates whether a preventive treatment was given during hospitalization prior to the culture and the culture was sensitive to it


In [ ]:
# ##emar
# microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df["emar_preemtive_medication"] = np.where(
#     microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.apply(
#         lambda row: any(med.lower() in row["ab_name"].lower() for med in str(row["emar_preemtive_medication"]).split())
#         and row["interpretation"] == "S",
#         axis=1
#     ),
#     1,
#     0
# )


# ##input_events
# microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df["ab_preemptive_ie"] = np.where(
#     microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.apply(
#         lambda row: (
#             isinstance(row["itemid_ab_inputevents_preemptive"], list) and  # Asegura que es una lista
#             isinstance(row["ab_name"], str) and  # Asegura que ab_name es string
#             any(row["ab_name"].lower() in str(med).lower() for med in row["itemid_ab_inputevents_preemptive"])  # Verifica coincidencias
#         ) and row["interpretation"] == "S",
#         axis=1
#     ),
#     1,
#     0
# )
# ##pyxis - no data

In [ ]:
# CHECKS
# microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df[microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['ab_preemptive_ie']==1][['ab_name','emar_preemtive_medication']]
# microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df[microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['ab_preemptive_ie']==1][['ab_name','itemid_ab_inputevents_preemptive']]

In [ ]:
# ### AB administered previous resistance
# microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['correct_empirical_ab'] = (
#     microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['ab_preemptive_ie'] +
#     microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['emar_preemtive_medication']
#     # + microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['ab_preemptive_pyxis']
# ).apply(lambda x: 1 if x >= 1 else 0)

In [ ]:
# microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df['correct_empirical_ab'].value_counts()

## Columns selection

In [ ]:
#Removing unnecesary repeated variables resulted from merges with other tables
columns_to_remove = [col for col in microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.columns if '_remove' in col]
microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.drop(columns=columns_to_remove, inplace=True)


In [ ]:
# microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.to_csv('/content/drive/MyDrive/FPS/PEANUT/analysis/data_outputs/hii.csv')

In [ ]:
print(microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df.columns.tolist())

In [ ]:
microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols = microbiology_icu_charlson_lab_emar_sofa_vitals_ie_py_df[[
'hadm_id',
'org_name',
'ab_name',
# 'charttime',
# 'microevent_id',
'interpretation',
# 'spec_type_desc',
# 'storetime',
'stay_id',
'gender',
# 'dod',
# 'dischtime',
'los_hospital',
'admission_age',
'race',
# 'hospital_expire_flag',
'hospstay_seq',
'first_hosp_stay',
# 'los_icu',
'icustay_seq',
# 'first_icu_stay',
# 'icu_intime',
# 'icu_outtime',
# 'subject_id_x',
# 'admittime',
# 'RowNum_x',
# 'subject_id_y',
# 'age_score',
'myocardial_infarct',
'congestive_heart_failure',
'peripheral_vascular_disease',
'cerebrovascular_disease',
'dementia',
'chronic_pulmonary_disease',
'rheumatic_disease',
'peptic_ulcer_disease',
'mild_liver_disease',
'diabetes_without_cc',
'diabetes_with_cc',
'paraplegia',
'renal_disease',
'malignant_cancer',
'severe_liver_disease',
'metastatic_solid_tumor',
'aids',
'charlson_comorbidity_index',
# 'charttime_blood_count',
# 'hematocrit',
# 'hemoglobin',
# 'mch',
# 'mchc',
# 'mcv',
# 'platelet',
# 'rbc',
# 'rdw',
# 'rdwsd',
# 'wbc',
# 'charttime_coagulation',
# 'd_dimer',
# 'fibrinogen',
# 'thrombin',
# 'inr',
# 'pt',
# 'ptt',
# 'subject_id',
# 'charttime_chemistry',
# 'specimen_id',
# 'albumin',
# 'globulin',
# 'total_protein',
# 'aniongap',
# 'bicarbonate',
# 'bun',
# 'calcium',
# 'chloride',
# 'creatinine',
# 'glucose',
# 'sodium',
# 'potassium',
# 'cultive_charttime',
# 'RowNum_y',
# 'emar_id',
# 'emar_seq',
# 'poe_id',
# 'pharmacy_id',
# 'enter_provider_id',
# 'event_txt',
# 'scheduletime',
# 'RowNum',
# 'amines_emar_presc',
# 'infection',
# 'explicit_sepsis',
# 'organ_dysfunction',
# 'mech_vent',
# 'ab_emar_presc',
# "emar_preemtive_medication",
# "itemid_ab_inputevents_preemptive",
# "ab_preemptive_ie",
# "med_rn",
# "name_ab_inputevents_pyxis",
# "gsn_rn",
# "gsn",
# "charttime_1",
# "stay_id_1",
# "subject_id_1",
# "ab_preemptive_pyxis",
# 'admission_location',
# 'hamd_id',
# 'seq_num',
# 'chartdate',
# 'icd_code',
# 'icd_version',
# 'hr',
# 'starttime_sofa',
# 'endtime',
# 'pao2fio2ratio_novent',
# 'pao2fio2ratio_vent',
# 'rate_epinephrine',
# 'rate_norepinephrine',
# 'rate_dopamine',
# 'rate_dobutamine',
# 'meanbp_min',
# 'gcs_min',
# 'uo_24hr',
# 'bilirubin_max',
# 'creatinine_max',
# 'platelet_min',
# 'respiration',
# 'coagulation',
'liver',
'cardiovascular',
'cns',
'renal',
# 'respiration_24hours',
# 'coagulation_24hours',
# 'liver_24hours',
# 'cardiovascular_24hours',
# 'cns_24hours',
# 'renal_24hours',
'sofa_24hours',
# 'charttime_vitalsign',
# 'heart_rate',
# 'sbp',
# 'dbp',
# 'mbp',
# 'sbp_ni',
# 'dbp_ni',
# 'mbp_ni',
# 'resp_rate',
# 'temperature',
# 'temperature_site',
# 'spo2',
# 'starttime',
# 'amount',
# 'rate',
# 'amines_ie_presc',
# 'ab_ie_presc',
# 'name',
"previous_icu_stays_5_years",
# 'weight_admit',
# 'height',
'BMI',
'betalactamic_allergy',
'sample_origin',
'day_of_week_icu_admission',
'month_icu_admission',
'day_of_week_culture_request',
'month_culture_request',
# 'block',
'has_previous_ab',
# 'ab_presc_total',
# 'group_ab',
# 'ab_group_Cefalosporinas de cuarta generacion',
# 'ab_group_Aminoglucósidos',
# 'ab_group_Monobactames',
# 'ab_group_Oxazolidonas',
# 'ab_group_Carbapenémicos',
# 'ab_group_Trimetoprim + Sulfonamida',
# 'ab_group_Cefalosporinas de tercera generacion',
# 'ab_group_Nitroimidazoles',
# 'ab_group_Quinolonas',
# 'ab_group_Tetraciclinas',
# 'ab_group_Penicilinas asociadas a inhibidores betalactamasa',
# 'ab_group_Lincosamidas',
# 'ab_group_Penicilinas',
# 'ab_group_Cefalosporinas de quinta generacion',
# 'ab_group_Macrólidos',
# 'ab_group_Cefalosporinas de primera generacion',
# 'active_principle',
# 'active_principleCefalosporinas de cuarta generacion',
# 'active_principleAminoglucósidos',
# 'active_principleMonobactames',
# 'active_principleOxazolidonas',
# 'active_principleCarbapenémicos',
# 'active_principleTrimetoprim + Sulfonamida',
# 'active_principleCefalosporinas de tercera generacion',
# 'active_principleNitroimidazoles',
# 'active_principleQuinolonas',
# 'active_principleTetraciclinas',
# 'active_principlePenicilinas asociadas a inhibidores betalactamasa',
# 'active_principleLincosamidas',
# 'active_principlePenicilinas',
# 'active_principleCefalosporinas de quinta generacion',
# 'active_principleMacrólidos',
# 'active_principleCefalosporinas de primera generacion',
'time_in_hospital',
'time_in_hospital_less_24h',
'oncologic_patient',
'hosp_before_90_days',
'stay_before_same_admission',
# 'stay_before',
'had_R_before_1_year',
# 'had_R_before_block',
'had_R_before_sameblock',
'had_R_before_block_aggregated',
'had_R_before_90_days',
'has_septic_shock',
'received_same_ab_before_test_str',
# "correct_empirical_ab"
]]



In [ ]:
columns_with_lists = [col for col in microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols.columns if microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols[col].apply(lambda x: isinstance(x, list)).any()]

print("Columnas que contienen listas:", columns_with_lists)

In [ ]:
# microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_EDA['active_principle'] = microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_EDA['active_principle'].apply(lambda x: ', '.join(x))
# microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_EDA['group_ab'] = microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_EDA['group_ab'].apply(lambda x: ', '.join(x))

## Columns transformation

In [ ]:
# 'WHITE'
microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols['race'] = np.where(
    microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols['race'].isin([
        'WHITE', 'WHITE - OTHER EUROPEAN', 'WHITE - RUSSIAN',
        'WHITE - BRAZILIAN', 'WHITE - EASTERN EUROPEAN'
    ]),
    'WHITE',
    microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols['race']
)

# 'BLACK/AFRICAN AMERICAN/AFRICAN'
microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols['race'] = np.where(
    microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols['race'].isin([
        'BLACK/AFRICAN AMERICAN', 'BLACK/AFRICAN'
    ]),
    'BLACK/AFRICAN AMERICAN/AFRICAN',
    microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols['race']
)
# Categorizar cualquier otra raza como 'OTHERS'
microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols['race'] = np.where(
    ~microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols['race'].isin([
        'BLACK/AFRICAN AMERICAN/AFRICAN', 'WHITE', 'UNKNOWN'
    ]),
    'OTHERS',
    microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols['race']
)


In [ ]:
microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols['interpretation'] = np.where(microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols['interpretation']=='R',1,0)

In [ ]:
microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols = microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols.rename(columns={'subject_id_x':'subject_id'})

In [ ]:
# cleaning admission weight variable we will replace by BMI
# microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols['weight_admit'] = microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols['weight_admit'].apply(lambda x: np.nan if x < 30 else x)
# microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols[microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols['weight_admit'].isna()][['subject_id','weight_admit']]
# microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols[microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols['weight_admit'].isna()][['hadm_id','weight_admit']]

#### To string

In [ ]:
list_to_str=[
'hadm_id',
'org_name',
'ab_name',
# 'charttime',
# 'microevent_id',
# 'interpretation',
# 'spec_type_desc',
# 'storetime',
# 'stay_id',
'gender',
# 'dod',
# 'dischtime',
# 'los_hospital',
# 'admission_age',
'race',
# 'hospital_expire_flag',
# 'hospstay_seq',
# 'first_hosp_stay',
# 'los_icu',
# 'icustay_seq',
# 'first_icu_stay',
# 'icu_intime',
# 'icu_outtime',
# 'subject_id_x',
# 'admittime',
# 'RowNum_x',
# 'subject_id_y',
# 'age_score',
# 'myocardial_infarct',
# 'congestive_heart_failure',
# 'peripheral_vascular_disease',
# 'cerebrovascular_disease',
# 'dementia',
# 'chronic_pulmonary_disease',
# 'rheumatic_disease',
# 'peptic_ulcer_disease',
# 'mild_liver_disease',
# 'diabetes_without_cc',
# 'diabetes_with_cc',
# 'paraplegia',
# 'renal_disease',
# 'malignant_cancer',
# 'severe_liver_disease',
# 'metastatic_solid_tumor',
# 'aids',
'charlson_comorbidity_index',
# 'charttime_blood_count',
# 'hematocrit',
# 'hemoglobin',
# 'mch',
# 'mchc',
# 'mcv',
# 'platelet',
# 'rbc',
# 'rdw',
# 'rdwsd',
# 'wbc',
# 'charttime_coagulation',
# 'd_dimer',
# 'fibrinogen',
# 'thrombin',
# 'inr',
# 'pt',
# 'ptt',
# 'subject_id',
# 'charttime_chemistry',
# 'specimen_id',
# 'albumin',
# 'globulin',
# 'total_protein',
# 'aniongap',
# 'bicarbonate',
# 'bun',
# 'calcium',
# 'chloride',
# 'creatinine',
# 'glucose',
# 'sodium',
# 'potassium',
# 'cultive_charttime',
# 'RowNum_y',
# 'emar_id',
# 'emar_seq',
# 'poe_id',
# 'pharmacy_id',
# 'enter_provider_id',
# 'event_txt',
# 'scheduletime',
# 'RowNum',
# 'amines_emar_presc',
# 'infection',
# 'explicit_sepsis',
# 'organ_dysfunction',
# 'mech_vent',
# 'ab_emar_presc',
# "emar_preemtive_medication",
# "itemid_ab_inputevents_preemptive",
# "ab_preemptive_ie",
# "med_rn",
# "name_ab_inputevents_pyxis",
# "gsn_rn",
# "gsn",
# "charttime_1",
# "stay_id_1",
# "subject_id_1",
# "ab_preemptive_pyxis",
# 'admission_location',
# 'hamd_id',
# 'seq_num',
# 'chartdate',
# 'icd_code',
# 'icd_version',
# 'hr',
# 'starttime_sofa',
# 'endtime',
# 'pao2fio2ratio_novent',
# 'pao2fio2ratio_vent',
# 'rate_epinephrine',
# 'rate_norepinephrine',
# 'rate_dopamine',
# 'rate_dobutamine',
# 'meanbp_min',
# 'gcs_min',
# 'uo_24hr',
# 'bilirubin_max',
# 'creatinine_max',
# 'platelet_min',
# 'respiration',
# 'coagulation',
# 'liver',
# 'cardiovascular',
# 'cns',
# 'renal',
# 'respiration_24hours',
# 'coagulation_24hours',
# 'liver_24hours',
# 'cardiovascular_24hours',
# 'cns_24hours',
# 'renal_24hours',
# 'sofa_24hours',
# 'charttime_vitalsign',
# 'heart_rate',
# 'sbp',
# 'dbp',
# 'mbp',
# 'sbp_ni',
# 'dbp_ni',
# 'mbp_ni',
# 'resp_rate',
# 'temperature',
# 'temperature_site',
# 'spo2',
# 'starttime',
# 'amount',
# 'rate',
# 'amines_ie_presc',
# 'ab_ie_presc',
# 'name',
# "previous_icu_stays_5_years",
# 'weight_admit',
# 'height',
# 'BMI',
# 'betalactamic_allergy',
'sample_origin',
'day_of_week_icu_admission',
'month_icu_admission',
'day_of_week_culture_request',
'month_culture_request',
# 'block',
# 'has_previous_ab',
# 'ab_presc_total',
# 'group_ab',
# 'ab_group_Cefalosporinas de cuarta generacion',
# 'ab_group_Aminoglucósidos',
# 'ab_group_Monobactames',
# 'ab_group_Oxazolidonas',
# 'ab_group_Carbapenémicos',
# 'ab_group_Trimetoprim + Sulfonamida',
# 'ab_group_Cefalosporinas de tercera generacion',
# 'ab_group_Nitroimidazoles',
# 'ab_group_Quinolonas',
# 'ab_group_Tetraciclinas',
# 'ab_group_Penicilinas asociadas a inhibidores betalactamasa',
# 'ab_group_Lincosamidas',
# 'ab_group_Penicilinas',
# 'ab_group_Cefalosporinas de quinta generacion',
# 'ab_group_Macrólidos',
# 'ab_group_Cefalosporinas de primera generacion',
# 'active_principle',
# 'active_principleCefalosporinas de cuarta generacion',
# 'active_principleAminoglucósidos',
# 'active_principleMonobactames',
# 'active_principleOxazolidonas',
# 'active_principleCarbapenémicos',
# 'active_principleTrimetoprim + Sulfonamida',
# 'active_principleCefalosporinas de tercera generacion',
# 'active_principleNitroimidazoles',
# 'active_principleQuinolonas',
# 'active_principleTetraciclinas',
# 'active_principlePenicilinas asociadas a inhibidores betalactamasa',
# 'active_principleLincosamidas',
# 'active_principlePenicilinas',
# 'active_principleCefalosporinas de quinta generacion',
# 'active_principleMacrólidos',
# 'active_principleCefalosporinas de primera generacion',
# 'time_in_hospital',
# 'time_in_hospital_less_24h',
# 'oncologic_patient',
# 'hosp_before_90_days',
# 'stay_before_same_admission',
# 'stay_before',
# 'had_R_before_1_year',
# 'had_R_before_block',
# 'had_R_before_sameblock',
'had_R_before_block_aggregated',
# 'had_R_before_90_days',
# 'has_septic_shock',
# 'received_same_ab_before_test_str',
# "correct_empirical_ab" --discarted variable
  ]

In [ ]:
microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols[list_to_str] = microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols[list_to_str].astype(str)

#### To num

In [ ]:
# ##columns with nans not recognized by tableone
list_to_num = [
# 'hadm_id',
# 'org_name',
# 'ab_name',
# 'charttime',
# 'microevent_id',
'interpretation',
# 'spec_type_desc',
# 'storetime',
# 'stay_id',
# 'gender',
# 'dod',
# 'dischtime',
'los_hospital',
'admission_age',
'race',
# 'hospital_expire_flag',
'hospstay_seq',
'first_hosp_stay',
# 'los_icu',
'icustay_seq',
# 'first_icu_stay',
# 'icu_intime',
# 'icu_outtime',
# 'subject_id_x',
# 'admittime',
# 'RowNum_x',
# 'subject_id_y',
# 'age_score',
'myocardial_infarct',
'congestive_heart_failure',
'peripheral_vascular_disease',
'cerebrovascular_disease',
'dementia',
'chronic_pulmonary_disease',
'rheumatic_disease',
'peptic_ulcer_disease',
'mild_liver_disease',
'diabetes_without_cc',
'diabetes_with_cc',
'paraplegia',
'renal_disease',
'malignant_cancer',
'severe_liver_disease',
'metastatic_solid_tumor',
'aids',
'charlson_comorbidity_index',
# 'charttime_blood_count',
# 'hematocrit',
# 'hemoglobin',
# 'mch',
# 'mchc',
# 'mcv',
# 'platelet',
# 'rbc',
# 'rdw',
# 'rdwsd',
# 'wbc',
# 'charttime_coagulation',
# 'd_dimer',
# 'fibrinogen',
# 'thrombin',
# 'inr',
# 'pt',
# 'ptt',
# 'subject_id',
# 'charttime_chemistry',
# 'specimen_id',
# 'albumin',
# 'globulin',
# 'total_protein',
# 'aniongap',
# 'bicarbonate',
# 'bun',
# 'calcium',
# 'chloride',
# 'creatinine',
# 'glucose',
# 'sodium',
# 'potassium',
# 'cultive_charttime',
# 'RowNum_y',
# 'emar_id',
# 'emar_seq',
# 'poe_id',
# 'pharmacy_id',
# 'enter_provider_id',
# 'event_txt',
# 'scheduletime',
# 'RowNum',
# 'amines_emar_presc',
# 'infection',
# 'explicit_sepsis',
# 'organ_dysfunction',
# 'mech_vent',
# 'ab_emar_presc',
# "emar_preemtive_medication",
# "itemid_ab_inputevents_preemptive",
# "ab_preemptive_ie",
# "med_rn",
# "name_ab_inputevents_pyxis",
# "gsn_rn",
# "gsn",
# "charttime_1",
# "stay_id_1",
# "subject_id_1",
# "ab_preemptive_pyxis",
# 'admission_location',
# 'hamd_id',
# 'seq_num',
# 'chartdate',
# 'icd_code',
# 'icd_version',
# 'hr',
# 'starttime_sofa',
# 'endtime',
# 'pao2fio2ratio_novent',
# 'pao2fio2ratio_vent',
# 'rate_epinephrine',
# 'rate_norepinephrine',
# 'rate_dopamine',
# 'rate_dobutamine',
# 'meanbp_min',
# 'gcs_min',
# 'uo_24hr',
# 'bilirubin_max',
# 'creatinine_max',
# 'platelet_min',
# 'respiration',
# 'coagulation',
'liver',
'cardiovascular',
'cns',
'renal',
# 'respiration_24hours',
# 'coagulation_24hours',
# 'liver_24hours',
# 'cardiovascular_24hours',
# 'cns_24hours',
# 'renal_24hours',
'sofa_24hours',
# 'charttime_vitalsign',
# 'heart_rate',
# 'sbp',
# 'dbp',
# 'mbp',
# 'sbp_ni',
# 'dbp_ni',
# 'mbp_ni',
# 'resp_rate',
# 'temperature',
# 'temperature_site',
# 'spo2',
# 'starttime',
# 'amount',
# 'rate',
# 'amines_ie_presc',
# 'ab_ie_presc',
# 'name',
"previous_icu_stays_5_years",
# 'weight_admit',
# 'height',
'BMI',
'betalactamic_allergy',
# 'sample_origin',
# 'day_of_week_icu_admission',
# 'month_icu_admission',
# 'day_of_week_culture_request',
# 'month_culture_request',
# 'block',
'has_previous_ab',
# 'ab_presc_total',
# 'group_ab',
# 'ab_group_Cefalosporinas de cuarta generacion',
# 'ab_group_Aminoglucósidos',
# 'ab_group_Monobactames',
# 'ab_group_Oxazolidonas',
# 'ab_group_Carbapenémicos',
# 'ab_group_Trimetoprim + Sulfonamida',
# 'ab_group_Cefalosporinas de tercera generacion',
# 'ab_group_Nitroimidazoles',
# 'ab_group_Quinolonas',
# 'ab_group_Tetraciclinas',
# 'ab_group_Penicilinas asociadas a inhibidores betalactamasa',
# 'ab_group_Lincosamidas',
# 'ab_group_Penicilinas',
# 'ab_group_Cefalosporinas de quinta generacion',
# 'ab_group_Macrólidos',
# 'ab_group_Cefalosporinas de primera generacion',
# 'active_principle',
# 'active_principleCefalosporinas de cuarta generacion',
# 'active_principleAminoglucósidos',
# 'active_principleMonobactames',
# 'active_principleOxazolidonas',
# 'active_principleCarbapenémicos',
# 'active_principleTrimetoprim + Sulfonamida',
# 'active_principleCefalosporinas de tercera generacion',
# 'active_principleNitroimidazoles',
# 'active_principleQuinolonas',
# 'active_principleTetraciclinas',
# 'active_principlePenicilinas asociadas a inhibidores betalactamasa',
# 'active_principleLincosamidas',
# 'active_principlePenicilinas',
# 'active_principleCefalosporinas de quinta generacion',
# 'active_principleMacrólidos',
# 'active_principleCefalosporinas de primera generacion',
'time_in_hospital',
'time_in_hospital_less_24h',
'oncologic_patient',
'hosp_before_90_days',
'stay_before_same_admission',
# 'stay_before',
'had_R_before_1_year',
# 'had_R_before_block',
'had_R_before_sameblock',
# 'had_R_before_block_aggregated',
'had_R_before_90_days',
'has_septic_shock',
'received_same_ab_before_test_str',
# "correct_empirical_ab"
]



In [ ]:
microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols[list_to_num] = (
    microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols[list_to_num]
    .apply(pd.to_numeric, errors='coerce'))

In [ ]:
microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols.replace('<NA>', pd.NA, inplace=True)

### Rename columns

In [ ]:
# new_column_names = {
#     'gender': 'sex',
#     'hospstay_seq': 'Number of Hospital Admissions before suspicion',
#     'icu_stays': 'Number of ICU Stays',
#     'time_in_hospital': 'Minutes in Hospital Before ICU admission',
#     'time_in_hospital_less_24h': 'Less than One Day in Hospital Before ICU admission',
#     'weight_admit' : 'Weight on hospital admission',
#     'had_R_before': 'MDRO detected Within One Year Before Suspicion',
#     'has_ab_prev': 'Antibiotics treatment Within 90 days prior suspicion',
#     'has_septic_shock': 'Amines treatment Within 6 hour prior suspicion',
# 'myocardial_infarct': 'Myocardial Infarct',
#     'congestive_heart_failure': 'Congestive Heart Failure',
#     'peripheral_vascular_disease': 'Peripheral Vascular Disease',
#     'cerebrovascular_disease': 'Cerebrovascular Disease',
#     'dementia': 'Dementia',
#     'chronic_pulmonary_disease': 'Chronic Pulmonary Disease',
#     'rheumatic_disease': 'Rheumatic Disease',
#     'peptic_ulcer_disease': 'Peptic Ulcer Disease',
#     'mild_liver_disease': 'Mild Liver Disease',
#     'diabetes_without_cc': 'Diabetes Without Complications',
#     'diabetes_with_cc': 'Diabetes With Complications',
#     'paraplegia': 'Paraplegia',
#     'renal_disease': 'Renal Disease',
#     'malignant_cancer': 'Malignant Cancer',
#     'severe_liver_disease': 'Severe Liver Disease',
#     'metastatic_solid_tumor': 'Metastatic Solid Tumor',
#     'aids': 'AIDS',
#     'charlson_comorbidity_index': 'Charlson Comorbidity Index',
#     'had_R_before_sameblock': 'MDRO Detected in Same Block Before Suspicion',
#     'had_R_before_block_aggregated': 'MDRO Detected in Aggregated Blocks Before Suspicion',
#     'has_septic_shock': 'Septic Shock Present',
#     'stay_before': 'ICU Stay one year previous this admission',
#     'hosp_before' : 'Hospital admission 90 days before the ICU admission',
#     'has_previous_ab' : 'Antibiotic administered previous resistance 90 days suspicion',
#     'previous_icu_stays_5_years' : 'Previous icu stays in the last 5 years'
# }

In [ ]:
# microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols = microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols.rename(columns=new_column_names)

In [ ]:
# microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols['correct_empirical_ab'].value_counts()

### One row by stay_id

We will select by stay id the first cultive results according to laboratory, urine output etc and taking into account the previous ab prescribed

In [ ]:
microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols = microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols.drop(columns=['ab_name','org_name'])

In [ ]:
cols_to_convert = [
    'oncologic_patient',
    'had_R_before_sameblock',
    'has_septic_shock',
    'received_same_ab_before_test_str',
    'has_previous_ab'
]

# Convert columns to boolean (0 -> False, 1 -> True)
microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols[cols_to_convert] = (
    microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols[cols_to_convert].astype(bool)
)

# If at least one row for a stay_id is True, set all rows for that stay_id to True
for col in cols_to_convert:
    microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols[col] = (
        microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols
        .groupby('stay_id')[col]
        .transform('max')
    )


In [ ]:
##if one cultive is Resistant the output will be labeled like it and received_same_ab_before_test_str TOO
microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id = microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols.loc[microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols.groupby(['stay_id'])['interpretation'].idxmax()].reset_index(drop = True)

### Dimensionality reduction

#### EDA

In [ ]:
microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id = microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id.drop(columns=['stay_id','hadm_id'])

In [ ]:
columns_with_lists = [col for col in microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id.columns if microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id[col].apply(lambda x: isinstance(x, list)).any()]

print("Columnas que contienen listas:")
print(columns_with_lists)

In [ ]:
# Select boolean columns
boolean_columns = microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id.select_dtypes(include='bool').columns

# Convert boolean columns to integers due to summaryDF library
microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id[boolean_columns] = microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id[boolean_columns].astype(int)

In [ ]:
# dfSummary(microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id)

#### Variables with more than 50% of missing deletion

In [ ]:
percent_missing = microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id.isnull().sum() * 100 / len(microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id)
missing_value_df = pd.DataFrame({'column_name': microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id.columns,
                                 'percent_missing': percent_missing})
missing_value_df.reset_index(drop=True, inplace=True)

In [ ]:
# Below code gives percentage of null in every column
null_percentage = microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id.isnull().sum()/microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id.shape[0]*100

# Below code gives list of columns having more than 50% null
col_to_drop = null_percentage[null_percentage>50].keys()
print('variables eliminadas', col_to_drop)
microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id_deleted_missing = microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id.drop(col_to_drop, axis=1)

In [ ]:
dfSummary(microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id_deleted_missing)

In [ ]:
microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_deleted_missing_without_imputation = microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id_deleted_missing.copy()

In [ ]:
# # Save df  with missing
# microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_deleted_missing_without_imputation.to_excel('/content/drive/MyDrive/FPS/PEANUT/analysis/data_outputs/microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_deleted_missing_without_imputation.xlsx', encoding='utf-8', index=False)

# microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_deleted_missing_without_imputation.to_csv('microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_deleted_missing_without_imputation_14082024.csv')
# files.download("microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_deleted_missing_without_imputation_14082024.csv")

#### ME QUEDO AQUI:Variables with high variance - me quitaria hasta el outcome

In [ ]:
# import pandas as pd
# from sklearn.feature_selection import VarianceThreshold

# # Compute the variance of each column
# variances = microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id_deleted_missing.var()

# # Define a threshold for low variance to identify near-constant features
# threshold = 0.0  # Adjust this threshold as needed

# # Filter columns with variance above the threshold (removing near-constant ones)
# filtered_columns = variances[variances > threshold]

# print("Columns retained (variance greater than {}):".format(threshold))
# print(filtered_columns.index)



###  Variable imputation - importante: ##check weight fill with the same patient

#### Variable **MICE** imputation: ME QUEDO AQUI

In [ ]:
# microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_mice_imputed = mice_imputation(microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_deleted_missing_without_imputation)

## Save resulted dataframes

In [ ]:
# joblib.dump(microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id, '/content/drive/MyDrive/FPS/PEANUT/analysis/data_outputs/microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_not_imputed_with_storetime.joblib')

In [ ]:
# microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id = microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id.drop(columns=['storetime','stay_id'])

#### Not by stay_id

In [ ]:
# microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_deleted_missing_without_imputation.to_excel('/content/drive/MyDrive/FPS/PEANUT/analysis/data_outputs/microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_deleted_missing_without_imputation.xlsx', encoding='utf-8', index=False)


In [ ]:
# joblib.dump(microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id, '/content/drive/MyDrive/FPS/PEANUT/analysis/data_outputs/microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_not_imputed.joblib')

#### By stay_id

In [ ]:
# microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_deleted_missing_without_imputation.to_excel(
#     '/content/drive/MyDrive/FPS/PEANUT/analysis/data_outputs/24032025_microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_deleted_missing_without_imputation.xlsx', encoding='utf-8', index=False)


In [ ]:
# microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_deleted_missing_without_imputation.to_csv(
#     '/content/drive/MyDrive/FPS/PEANUT/analysis/data_outputs/24032025_microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_deleted_missing_without_imputation.csv',
#     encoding='utf-8',
#     index=False
# )

In [ ]:
microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_deleted_missing_without_imputation

In [ ]:
# joblib.dump(microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id_deleted_missing, '/content/drive/MyDrive/FPS/PEANUT/analysis/data_outputs/161224_microbiology_icu_charlson_lab_sofa_vitals_df_selected_cols_by_stay_id_deleted_missing.joblib')

#### Cleaned with imputations

In [ ]:
# joblib.dump(microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_mice_imputed, '/content/drive/MyDrive/FPS/PEANUT/analysis/data_outputs/microbiology_icu_charlson_lab_sofa_vitals_urinedf_by_stay_id_mice_imputed.joblib')